# Project 03 · Neural Sequence Laboratory
## Session 2 — Build LSTM and GRU cells from their gates

The vanilla RNN rewrites its entire hidden state at every character. In this notebook you will replace that fragile update with two gated alternatives.

All data preparation, batching, model wrapping, optimization, evaluation, gradient diagnostics, comparison, and generation code is provided. You implement exactly two substantial pieces:

1. a complete **LSTM cell**; and
2. a complete **GRU cell**.

> **Success condition:** both behavioral checks pass, both models train, and both invent new names.

### Why you are not implementing backward passes

You are responsible for the complete forward computation of both gated cells. PyTorch autograd will construct BPTT from those operations. A supplied audit later plots gradients reaching earlier states, so you will inspect temporal credit without manually deriving dozens of gate derivatives.

In [ ]:
from __future__ import annotations

import math
import random
import time
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import TypeAlias

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 351
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"PyTorch {torch.__version__} · device={DEVICE}")

## 0 · Identical experiment setup — provided

The dataset, vocabulary, split seed, padding policy, embedding size, hidden size, optimizer, epoch count, and evaluation functions are shared across both models. Only the recurrent cell changes.

Dataset source: [karpathy/makemore — names.txt](https://github.com/karpathy/makemore/blob/master/names.txt).

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
DATA_PATH = Path("babynames.txt")
if not DATA_PATH.exists():
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

names = [line.strip().lower() for line in DATA_PATH.read_text().splitlines() if line.strip()]
assert len(names) == 32_033

PAD_TOKEN, BOS_TOKEN, EOS_TOKEN = "<PAD>", "<BOS>", "<EOS>"
characters = sorted(set("".join(names)))
itos = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN] + characters
stoi = {token: index for index, token in enumerate(itos)}
PAD_ID, BOS_ID, EOS_ID = stoi[PAD_TOKEN], stoi[BOS_TOKEN], stoi[EOS_TOKEN]
VOCAB_SIZE = len(itos)

def encode(text: str) -> list[int]:
    return [stoi[character] for character in text]

def decode(ids: list[int]) -> str:
    return "".join(itos[index] for index in ids if index >= 3)

def make_example(name: str) -> tuple[list[int], list[int]]:
    ids = encode(name)
    return [BOS_ID] + ids, ids + [EOS_ID]

print(f"{len(names):,} names · {VOCAB_SIZE} tokens")

In [ ]:
class NameDataset(Dataset):
    def __init__(self, items: list[str]):
        self.items = items

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, index: int) -> tuple[list[int], list[int]]:
        return make_example(self.items[index])

def collate_names(batch: list[tuple[list[int], list[int]]]) -> tuple[torch.Tensor, torch.Tensor]:
    width = max(len(inputs) for inputs, _ in batch)
    inputs = torch.full((len(batch), width), PAD_ID, dtype=torch.long)
    targets = torch.full((len(batch), width), PAD_ID, dtype=torch.long)
    for row, (input_ids, target_ids) in enumerate(batch):
        inputs[row, :len(input_ids)] = torch.tensor(input_ids)
        targets[row, :len(target_ids)] = torch.tensor(target_ids)
    return inputs, targets

shuffled = names.copy()
random.Random(SEED).shuffle(shuffled)
n_train, n_valid = int(.8 * len(shuffled)), int(.1 * len(shuffled))
train_names = shuffled[:n_train]
valid_names = shuffled[n_train:n_train + n_valid]
test_names = shuffled[n_train + n_valid:]

BATCH_SIZE = 256
train_loader = DataLoader(NameDataset(train_names), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_names, generator=torch.Generator().manual_seed(SEED))
valid_loader = DataLoader(NameDataset(valid_names), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_names)
test_loader = DataLoader(NameDataset(test_names), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_names)
print(f"split: {len(train_names):,} / {len(valid_names):,} / {len(test_names):,}")

## Question 1 · Implement the complete LSTM cell

Your cell receives an embedded character $e_t$ and state $(h_{t-1}, c_{t-1})$. Use one affine projection from $[e_t;h_{t-1}]$ to four vectors of size `hidden_dim`, then apply:

$$\begin{aligned}
f_t &= \sigma(\cdot), & i_t &= \sigma(\cdot),\\
g_t &= \tanh(\cdot), & o_t &= \sigma(\cdot),\\
c_t &= f_t \odot c_{t-1} + i_t \odot g_t,\\
h_t &= o_t \odot \tanh(c_t).
\end{aligned}$$

### Required implementation

Complete all three methods. You choose and create the projection layer, initialize both state tensors, split the gate logits in the documented order, apply nonlinearities, and return the new state.

| Projection slice | Nonlinearity | Meaning |
|---|---|---|
| 1 | sigmoid | forget gate $f_t$ |
| 2 | sigmoid | input gate $i_t$ |
| 3 | tanh | candidate $g_t$ |
| 4 | sigmoid | output gate $o_t$ |

In [ ]:
LSTMState: TypeAlias = tuple[torch.Tensor, torch.Tensor]

class ManualLSTMCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim

        # TODO Q1a — create one affine projection to 4 * hidden_dim outputs.
        self.gate_projection = ...

    def initial_state(self, batch_size: int, device: torch.device) -> LSTMState:
        # TODO Q1b — return zero h_0 and c_0, each [batch, hidden_dim].
        ...

    def forward(self, embedded: torch.Tensor, state: LSTMState) -> tuple[torch.Tensor, LSTMState]:
        previous_hidden, previous_cell = state

        # TODO Q1c — concatenate, project, split, activate, and update c_t and h_t.
        combined = ...
        gate_logits = ...
        forget_logits, input_logits, candidate_logits, output_logits = ...

        forget_gate = ...
        input_gate = ...
        candidate = ...
        output_gate = ...

        cell = ...
        hidden = ...
        return hidden, (hidden, cell)

### Behavioral check: can the cell preserve memory?

The check sets all weights to zero and chooses biases that make $f\approx1$, $i\approx0$, and $o\approx1$. The new cell state should closely preserve the old cell state. This tests behavior rather than only tensor shapes.

In [ ]:
# Check Q1
lstm_probe = ManualLSTMCell(input_dim=3, hidden_dim=2)
assert sum(parameter.numel() for parameter in lstm_probe.parameters()) == (3 + 2) * (4 * 2) + 4 * 2
with torch.no_grad():
    lstm_probe.gate_projection.weight.zero_()
    # order: forget, input, candidate, output
    lstm_probe.gate_projection.bias.copy_(torch.tensor([10., 10., -10., -10., 0., 0., 10., 10.]))
previous_cell = torch.tensor([[0.7, -0.4]])
previous_hidden = torch.zeros(1, 2)
hidden, (_, cell) = lstm_probe(torch.zeros(1, 3), (previous_hidden, previous_cell))
assert hidden.shape == cell.shape == (1, 2)
assert torch.allclose(cell, previous_cell, atol=2e-4), (cell, previous_cell)
assert torch.allclose(hidden, torch.tanh(previous_cell), atol=2e-4)
print("✓ Q1 passed: the LSTM can preserve a supplied memory state.")

## Question 2 · Implement the complete GRU cell

The GRU keeps one state. Use one projection from $[e_t;h_{t-1}]$ to reset and update gates, and a second projection from $[e_t;r_t\odot h_{t-1}]$ to the candidate:

$$\begin{aligned}
r_t &= \sigma(\cdot),\\
z_t &= \sigma(\cdot),\\
\widetilde h_t &= \tanh(W_h[e_t;r_t\odot h_{t-1}] + b_h),\\
h_t &= (1-z_t)\odot h_{t-1} + z_t\odot\widetilde h_t.
\end{aligned}$$

Complete the constructor, zero-state creation, both gates, candidate, and interpolation. The gate projection order must be **reset, update**.

In [ ]:
class ManualGRUCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim

        # TODO Q2a — create the reset/update projection and candidate projection.
        self.gate_projection = ...
        self.candidate_projection = ...

    def initial_state(self, batch_size: int, device: torch.device) -> torch.Tensor:
        # TODO Q2b — return zero h_0 shaped [batch, hidden_dim].
        ...

    def forward(self, embedded: torch.Tensor, previous_hidden: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO Q2c — gates, reset state, candidate, and final interpolation.
        combined = ...
        gate_logits = ...
        reset_logits, update_logits = ...
        reset_gate = ...
        update_gate = ...

        candidate_input = ...
        candidate = ...
        hidden = ...
        return hidden, hidden

### Behavioral check: does the update gate interpolate?

With a zero candidate, $z\approx0$ should preserve the old state and $z\approx1$ should replace it with approximately zero.

In [ ]:
# Check Q2
gru_probe = ManualGRUCell(input_dim=3, hidden_dim=2)
with torch.no_grad():
    gru_probe.gate_projection.weight.zero_()
    gru_probe.candidate_projection.weight.zero_()
    gru_probe.candidate_projection.bias.zero_()
    # reset bias is irrelevant here; update bias is very negative.
    gru_probe.gate_projection.bias.copy_(torch.tensor([0., 0., -10., -10.]))
previous = torch.tensor([[0.7, -0.4]])
hidden_keep, _ = gru_probe(torch.zeros(1, 3), previous)
assert torch.allclose(hidden_keep, previous, atol=2e-4)
with torch.no_grad():
    gru_probe.gate_projection.bias[2:].fill_(10.)
hidden_replace, _ = gru_probe(torch.zeros(1, 3), previous)
assert torch.allclose(hidden_replace, torch.zeros_like(previous), atol=2e-4)
print("✓ Q2 passed: the GRU update gate interpolates old and candidate states.")

## 1 · Shared language-model wrapper — provided

The wrapper embeds each character, repeatedly calls your cell, and maps the exposed hidden state to vocabulary logits. It works with either a tensor state (GRU) or tuple state (LSTM).

In [ ]:
State: TypeAlias = torch.Tensor | LSTMState

class CharacterGatedLM(nn.Module):
    def __init__(self, cell: nn.Module, embedding_dim: int, hidden_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(VOCAB_SIZE, embedding_dim, padding_idx=PAD_ID)
        self.cell = cell
        self.output = nn.Linear(hidden_dim, VOCAB_SIZE)

    def initial_state(self, batch_size: int, device: torch.device) -> State:
        return self.cell.initial_state(batch_size, device)

    def step(self, token_ids: torch.Tensor, state: State) -> tuple[torch.Tensor, State]:
        embedded = self.embedding(token_ids)
        hidden, state = self.cell(embedded, state)
        return self.output(hidden), state

    def forward(self, token_ids: torch.Tensor, retain_states: bool = False) -> tuple[torch.Tensor, list[torch.Tensor]]:
        batch_size, time_steps = token_ids.shape
        state = self.initial_state(batch_size, token_ids.device)
        logits_by_time = []
        exposed_states = []
        for time_step in range(time_steps):
            logits, state = self.step(token_ids[:, time_step], state)
            hidden = state[0] if isinstance(state, tuple) else state
            if retain_states:
                hidden.retain_grad()
                exposed_states.append(hidden)
            logits_by_time.append(logits)
        return torch.stack(logits_by_time, dim=1), exposed_states

EMBEDDING_DIM = 32
HIDDEN_DIM = 96

def build_lstm() -> CharacterGatedLM:
    return CharacterGatedLM(ManualLSTMCell(EMBEDDING_DIM, HIDDEN_DIM), EMBEDDING_DIM, HIDDEN_DIM).to(DEVICE)

def build_gru() -> CharacterGatedLM:
    return CharacterGatedLM(ManualGRUCell(EMBEDDING_DIM, HIDDEN_DIM), EMBEDDING_DIM, HIDDEN_DIM).to(DEVICE)

## 2 · Training and evaluation — provided

Both models receive the same optimizer settings and global-norm clipping. Training uses ordinary autograd: your forward equations define the graph, and `.backward()` applies BPTT end to end.

In [ ]:
loss_function = nn.CrossEntropyLoss(ignore_index=PAD_ID)

def sequence_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return loss_function(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

@torch.no_grad()
def evaluate(model: CharacterGatedLM, loader: DataLoader) -> float:
    model.eval()
    loss_sum, token_count = 0.0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        logits, _ = model(inputs)
        loss = sequence_loss(logits, targets)
        count = targets.ne(PAD_ID).sum().item()
        loss_sum += loss.item() * count
        token_count += count
    return loss_sum / token_count

@dataclass
class TrainingResult:
    model_name: str
    model: CharacterGatedLM
    train_losses: list[float]
    validation_losses: list[float]
    seconds: float

def train(model_name: str, model: CharacterGatedLM, epochs: int = 12) -> TrainingResult:
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)
    train_losses, validation_losses = [], []
    started = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum, token_count = 0.0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(inputs)
            loss = sequence_loss(logits, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            count = targets.ne(PAD_ID).sum().item()
            loss_sum += loss.item() * count
            token_count += count
        train_loss = loss_sum / token_count
        validation_loss = evaluate(model, valid_loader)
        train_losses.append(train_loss)
        validation_losses.append(validation_loss)
        print(f"{model_name:4s} · epoch {epoch:02d} · train {train_loss:.3f} · valid {validation_loss:.3f} · ppl {math.exp(validation_loss):.2f}")
    return TrainingResult(model_name, model, train_losses, validation_losses, time.perf_counter() - started)

In [ ]:
torch.manual_seed(SEED)
lstm_result = train("LSTM", build_lstm(), epochs=12)
torch.manual_seed(SEED)
gru_result = train("GRU", build_gru(), epochs=12)

In [ ]:
plt.figure(figsize=(9, 4))
for result in (lstm_result, gru_result):
    plt.plot(range(1, len(result.validation_losses) + 1), result.validation_losses, marker="o", label=result.model_name)
plt.xlabel("epoch")
plt.ylabel("validation cross-entropy")
plt.title("Same task and budget; different gated cell")
plt.legend()
plt.show()

## 3 · Fair comparison — provided measurements, your interpretation

Parameter counts are not forced to be identical because the cells contain different numbers of projections. The table makes that difference explicit instead of hiding it.

In [ ]:
def parameter_count(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())

print(f"{'model':<8} {'parameters':>12} {'valid loss':>12} {'test ppl':>10} {'seconds':>10}")
print("-" * 58)
for result in (lstm_result, gru_result):
    test_loss = evaluate(result.model, test_loader)
    print(f"{result.model_name:<8} {parameter_count(result.model):>12,} {result.validation_losses[-1]:>12.3f} {math.exp(test_loss):>10.2f} {result.seconds:>10.1f}")

Answer briefly:

1. Which model reaches the lower validation loss?
2. How large is the parameter-count difference?
3. Is the comparison controlled but imperfect, or strictly equal? Explain.
4. Which cell would you choose for this short-name task, and what evidence supports the choice?

## 4 · Inspect BPTT without implementing it

The next supplied function creates a loss only at the final real character of each sequence. It retains intermediate hidden-state gradients and plots their mean norm. Earlier positions can receive credit only through the recurrent chain.

In [ ]:
def final_loss_gradient_profile(model: CharacterGatedLM, batch_size: int = 128) -> list[float]:
    model.train()
    inputs, targets = next(iter(DataLoader(NameDataset(valid_names), batch_size=batch_size, shuffle=False, collate_fn=collate_names)))
    inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
    model.zero_grad(set_to_none=True)
    logits, states = model(inputs, retain_states=True)
    lengths = targets.ne(PAD_ID).sum(dim=1)
    rows = torch.arange(inputs.size(0), device=DEVICE)
    final_positions = lengths - 1
    final_logits = logits[rows, final_positions]
    final_targets = targets[rows, final_positions]
    nn.functional.cross_entropy(final_logits, final_targets).backward()
    return [state.grad.norm(dim=1).mean().item() if state.grad is not None else 0.0 for state in states]

plt.figure(figsize=(9, 4))
for result in (lstm_result, gru_result):
    profile = final_loss_gradient_profile(result.model)
    plt.plot(range(len(profile)), profile, marker="o", label=result.model_name)
plt.yscale("log")
plt.xlabel("time step from the beginning")
plt.ylabel("mean hidden-state gradient norm · log scale")
plt.title("A final-token loss sends credit backward through the cell")
plt.legend()
plt.show()

### Gradient interpretation

1. Why is the horizontal axis a time axis rather than a layer-depth axis?
2. Why did we attach loss only to the final real token for this diagnostic?
3. Does a nonzero early-state gradient prove that the model learned a useful long dependency? Why not?

## 5 · Success moment: let both cells invent names

Generation is supplied here because the new learning objective was the gated cell itself. We use the exact same sampler for both models.

In [ ]:
@torch.no_grad()
def generate_name(model: CharacterGatedLM, temperature: float = .8, max_length: int = 20) -> str:
    model.eval()
    state = model.initial_state(1, DEVICE)
    current = torch.tensor([BOS_ID], device=DEVICE)
    generator = torch.Generator()
    generator.seed()
    result = []
    for _ in range(max_length):
        logits, state = model.step(current, state)
        probabilities = torch.softmax((logits / temperature).squeeze(0).cpu(), dim=-1)
        next_id = int(torch.multinomial(probabilities, 1, generator=generator).item())
        if next_id == EOS_ID:
            break
        if next_id not in (PAD_ID, BOS_ID):
            result.append(next_id)
        current = torch.tensor([next_id], device=DEVICE)
    return decode(result)

known = set(names)
for result in (lstm_result, gru_result):
    print(f"\n{result.model_name} · temperature 0.80")
    generated = []
    while len(generated) < 15:
        candidate = generate_name(result.model, temperature=.8)
        if candidate and candidate not in generated:
            generated.append(candidate)
    for candidate in generated:
        status = "NEW" if candidate not in known else "seen"
        print(f"  {candidate:<18} {status}")

## Final reflection

Write 200–300 words:

1. Explain the LSTM cell-state update in your own words.
2. Explain the GRU interpolation and the two extreme cases $z_t\approx0$ and $z_t\approx1$.
3. Compare validation loss, parameter count, training time, gradient profile, and generated samples.
4. Which gated cell would you carry into the longer recipe task, and why?

Before submission, restart the kernel and run all cells. Both `✓` checks, both learning curves, the gradient audit, and both name lists must be visible.